# 🚀 Notebook 03 — PPO Fine-tuning with LoRA/PEFT

**RLHF Preference Trainer** · Step 3 of 5

This notebook fine-tunes GPT-2 Medium using **Proximal Policy Optimization (PPO)** against the trained reward model.

Key techniques:
- **TRL** (`PPOTrainer`) for the PPO loop
- **LoRA** (rank=8, alpha=16) via PEFT for parameter-efficient training
- Only **~0.8%** of parameters are trainable → fits in 8GB VRAM
- KL penalty to prevent reward hacking

The PPO model is saved to `ppo_model/` for evaluation.

> **Runtime**: T4 GPU strongly recommended (~60–90 min for 200 PPO steps).

---

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q trl peft transformers torch accelerate bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Path setup ───────────────────────────────────────────────────
import os, sys

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('rlhf-preference-trainer'):
        !git clone https://github.com/sharma614/rlhf-preference-trainer.git
    os.chdir('rlhf-preference-trainer')
    # Copy reward_model from Drive if available
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        drive_rm = '/content/drive/MyDrive/rlhf_data/reward_model'
        if os.path.exists(drive_rm):
            !cp -r "{drive_rm}" .
            print("✅ Copied reward_model from Drive")
    except Exception as e:
        print(f"Drive: {e}")
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(project_root)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"📁 CWD: {os.getcwd()}")

In [ ]:
# ── Cell 3: Imports ──────────────────────────────────────────────────────
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from peft import get_peft_model, LoraConfig, TaskType
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl import create_reference_model

from src.reward_model import BradleyTerryRewardModel, score_response
from src.ppo_config import (
    MODEL_NAME, REWARD_MODEL_DIR, PPO_MODEL_DIR, SEED,
    LORA_R, LORA_ALPHA, LORA_DROPOUT, LORA_TARGET_MODULES, LORA_BIAS,
    PPO_LEARNING_RATE, PPO_BATCH_SIZE, PPO_MINI_BATCH_SIZE,
    PPO_GRADIENT_ACCUMULATION, PPO_EPOCHS, PPO_STEPS,
    PPO_INIT_KL_COEF, PPO_TARGET_KL, PPO_MAX_GRAD_NORM,
    MAX_NEW_TOKENS, SEED_PROMPTS as PROMPTS, trainable_param_percent
)
from src.data_utils import SEED_PROMPTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device: {device}")
if device == 'cpu':
    print("⚠️  CPU detected — PPO will be very slow. Reduced steps will run.")

In [ ]:
# ── Cell 4: Load tokenizer ───────────────────────────────────────────────
print(f"⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer ready")

In [ ]:
# ── Cell 5: Load base model and apply LoRA ───────────────────────────────
print(f"⏳ Loading {MODEL_NAME} and applying LoRA...")

# Base model for PPO (with value head)
base_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)

# LoRA configuration — matches resume: rank=8, alpha=16
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias=LORA_BIAS,
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the pretrained transformer within the value-head wrapper
base_model.pretrained_model = get_peft_model(base_model.pretrained_model, lora_config)

print(f"\n✅ LoRA applied — Configuration:")
print(f"   r (rank)         : {LORA_R}")
print(f"   alpha            : {LORA_ALPHA}")
print(f"   dropout          : {LORA_DROPOUT}")
print(f"   target_modules   : {LORA_TARGET_MODULES}")
print()
pct = trainable_param_percent(base_model)
print(f"   → Resume claims ~0.8%; we expect similar here.")

In [ ]:
# ── Cell 6: Load reward model ────────────────────────────────────────────
import os

if os.path.exists(REWARD_MODEL_DIR):
    print(f"⏳ Loading reward model from {REWARD_MODEL_DIR}/...")
    reward_model = BradleyTerryRewardModel.from_pretrained(REWARD_MODEL_DIR, device=device)
    reward_model.eval()
    print("✅ Reward model loaded")
else:
    print(f"⚠️  Reward model not found at {REWARD_MODEL_DIR}")
    print("   Using untrained reward model for demonstration.")
    print("   Run notebook 02 first for real RLHF training.")
    reward_model = BradleyTerryRewardModel(model_name=MODEL_NAME)
    reward_model = reward_model.to(device)
    reward_model.eval()

In [ ]:
# ── Cell 7: Reward function ──────────────────────────────────────────────
def compute_rewards(prompts_list, responses_list):
    """
    Score each (prompt, response) pair with the reward model.
    Returns a list of scalar torch tensors for PPOTrainer.
    """
    rewards = []
    with torch.no_grad():
        for prompt, response in zip(prompts_list, responses_list):
            score = score_response(
                reward_model, tokenizer, prompt, response,
                max_length=256, device=device
            )
            rewards.append(torch.tensor(float(score), dtype=torch.float32))
    return rewards

# Test the reward function
test_r = compute_rewards(
    ["Explain quantum computing."],
    ["Quantum computing uses quantum bits (qubits) that can exist in superposition."]
)
print(f"✅ Reward function test: {test_r[0].item():.4f}")

In [ ]:
# ── Cell 8: PPO configuration ────────────────────────────────────────────
# Reduce steps on CPU to avoid timeout
actual_steps = PPO_STEPS if device == 'cuda' else 20

ppo_config = PPOConfig(
    model_name=MODEL_NAME,
    learning_rate=PPO_LEARNING_RATE,
    batch_size=PPO_BATCH_SIZE,
    mini_batch_size=PPO_MINI_BATCH_SIZE,
    gradient_accumulation_steps=PPO_GRADIENT_ACCUMULATION,
    ppo_epochs=PPO_EPOCHS,
    init_kl_coef=PPO_INIT_KL_COEF,
    target=PPO_TARGET_KL,
    cliprange=0.2,
    vf_coef=0.1,
    max_grad_norm=PPO_MAX_GRAD_NORM,
    seed=SEED,
    log_with=None,  # Set to 'wandb' if you want W&B logging
)

print(f"PPO Configuration:")
print(f"  learning_rate : {PPO_LEARNING_RATE}")
print(f"  batch_size    : {PPO_BATCH_SIZE}")
print(f"  ppo_epochs    : {PPO_EPOCHS}")
print(f"  total steps   : {actual_steps} ({'GPU' if device=='cuda' else 'CPU demo'})")
print(f"  KL coef       : {PPO_INIT_KL_COEF} (target={PPO_TARGET_KL})")

In [ ]:
# ── Cell 9: Build PPOTrainer ─────────────────────────────────────────────
# Reference model (frozen copy) for KL divergence computation
ref_model = create_reference_model(base_model)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=base_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
)

print("✅ PPOTrainer initialized")

In [ ]:
# ── Cell 10: PPO Training Loop ───────────────────────────────────────────
import random, time

os.makedirs(PPO_MODEL_DIR, exist_ok=True)

generation_kwargs = {
    'max_new_tokens': MAX_NEW_TOKENS,
    'do_sample': True,
    'temperature': 0.9,
    'top_p': 0.92,
    'pad_token_id': tokenizer.eos_token_id,
    'repetition_penalty': 1.3,
}

ppo_log = []
rng = random.Random(SEED)

print(f"🚀 Starting PPO training for {actual_steps} steps...")
start_time = time.time()

for step in range(actual_steps):
    # Sample a batch of prompts
    batch_prompts = [rng.choice(SEED_PROMPTS) for _ in range(PPO_BATCH_SIZE)]

    # Tokenize prompts
    input_ids_list = [
        tokenizer.encode(p, return_tensors='pt', truncation=True, max_length=128).squeeze(0)
        for p in batch_prompts
    ]

    # Generate responses
    response_tensors = ppo_trainer.generate(
        input_ids_list,
        **generation_kwargs,
    )

    # Decode responses (only the newly generated tokens)
    batch_responses = [
        tokenizer.decode(
            resp[len(inp):],
            skip_special_tokens=True
        ).strip()
        for inp, resp in zip(input_ids_list, response_tensors)
    ]

    # Compute rewards
    rewards = compute_rewards(batch_prompts, batch_responses)

    # PPO update step
    stats = ppo_trainer.step(input_ids_list, response_tensors, rewards)

    # Logging
    mean_reward = float(torch.stack(rewards).mean())
    mean_kl = float(stats.get('ppo/mean_non_score_reward', 0))

    log_entry = {
        'step': step + 1,
        'mean_reward': mean_reward,
        'mean_kl': mean_kl,
        'policy_loss': float(stats.get('ppo/policy/loss', 0)),
        'value_loss': float(stats.get('ppo/value/loss', 0)),
    }
    ppo_log.append(log_entry)

    if (step + 1) % 10 == 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"  Step {step+1:3d}/{actual_steps} | "
            f"Reward: {mean_reward:+.4f} | "
            f"KL: {mean_kl:.4f} | "
            f"Elapsed: {elapsed:.1f}m"
        )

total_time = (time.time() - start_time) / 60
print(f"\n✅ PPO training complete in {total_time:.1f} minutes")

In [ ]:
# ── Cell 11: Save PPO log and model ─────────────────────────────────────
ppo_log_df = pd.DataFrame(ppo_log)
ppo_log_df.to_csv(f"{PPO_MODEL_DIR}/ppo_training_log.csv", index=False)
print(f"✅ PPO log saved to {PPO_MODEL_DIR}/ppo_training_log.csv")

# Save the LoRA model
base_model.pretrained_model.save_pretrained(PPO_MODEL_DIR)
tokenizer.save_pretrained(PPO_MODEL_DIR)

# Also save as merged (full model) for easy inference — optional, uses more disk
try:
    merged_model = base_model.pretrained_model.merge_and_unload()
    merged_model.save_pretrained(f"{PPO_MODEL_DIR}_merged")
    print(f"✅ Merged model saved to {PPO_MODEL_DIR}_merged/")
except Exception as e:
    print(f"⚠️  Merge failed (OK for demo): {e}")

print(f"\n✅ LoRA adapter saved to {PPO_MODEL_DIR}/")

In [ ]:
# ── Cell 12: Quick plot of PPO curves ────────────────────────────────────
if len(ppo_log_df) > 0:
    from src.evaluation import plot_ppo_curves
    os.makedirs('evaluation_results', exist_ok=True)
    plot_ppo_curves(
        ppo_log_df,
        save_path='evaluation_results/ppo_curves.png',
        show=True
    )
    print(f"\n  Final mean reward: {ppo_log_df['mean_reward'].tail(20).mean():.4f}")
    print(f"  Initial mean reward: {ppo_log_df['mean_reward'].head(20).mean():.4f}")

In [ ]:
# ── Cell 13: LoRA parameter verification ────────────────────────────────
print("LoRA Module Summary:")
lora_params = 0
for name, param in base_model.named_parameters():
    if 'lora_' in name and param.requires_grad:
        lora_params += param.numel()
        print(f"  {name:60s} {param.shape}")

total_params = sum(p.numel() for p in base_model.parameters())
print(f"\n  LoRA params  : {lora_params:,}")
print(f"  Total params : {total_params:,}")
print(f"  LoRA %       : {100.0 * lora_params / total_params:.2f}%")
print(f"  (Resume claims ~0.8% — expect similar)")

In [ ]:
# ── Cell 14: Smoke test ──────────────────────────────────────────────────
assert os.path.exists(PPO_MODEL_DIR), f"{PPO_MODEL_DIR} not found"
assert os.path.exists(f"{PPO_MODEL_DIR}/ppo_training_log.csv"), "Log missing"
assert len(ppo_log_df) > 0, "Empty PPO log"

print("✅ Smoke test PASSED")
print(f"   PPO ran for {len(ppo_log_df)} steps")
print(f"   Reward range: [{ppo_log_df['mean_reward'].min():.4f}, {ppo_log_df['mean_reward'].max():.4f}]")
print(f"\n   ✨ Next step: Run notebook 04_evaluation.ipynb")